In [ ]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms

import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.train import net_train_AnyNet_L, net_train_ViT_L, net_train_RNN_L, net_train_LC_L

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code
Library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [ ]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-a7f2d2a8-1917-e5b1-e7dd-ce7250d8fd73)


In [ ]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [ ]:
# @title Load data
file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
brain_signal_lfp = torch.load(file_dir + '/brain_signal_lfp1.pt')
for file_id in [2, 3, 4, 5]:
    brain_signal_lfp = torch.concat([brain_signal_lfp, torch.load(file_dir + f'/brain_signal_lfp{file_id}.pt')], dim=0)
    print(f'Load file id: {file_id}')

# torch.save(brain_signal_lfp, '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/brain_signal_lfp')
list_dict = torch.load(file_dir + '/list_dict.pt')
# list_dict_acronym_selec = torch.load(file_dir + '/list_dict_acronym_selec.pt')
brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']

Load file id: 2
Load file id: 3
Load file id: 4
Load file id: 5


In [ ]:
if len(brain_signal_lfp) == len(brain_region_index):
    print('Matched, no damage!')

Matched, no damage!


In [ ]:
save_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44'

In [ ]:
list_dict.keys()

dict_keys(['brain_region_index', 'brain_region_index_Cosmos', 'brain_region_atlas', 'subject_list', 'exp_list', 'key_list', 'coordinate_list', 'depth_list', 'volume_list', 'brain_signal_lfp', 'brain_signal_ap', 'train_list_key', 'test_list_key', 'train_list_subject', 'test_list_subject', 'train_list_exp', 'test_list_exp', 'train_list_trial', 'test_list_trial', 'train_list_intest', 'test_list_intest', 'acronym_selec_list', 'valid_list_intest'])

In [ ]:
subject_num = 10
key = 'stimOff_times'
subject_od_ind, subject_od_list = subject_od_ind_gen(list_dict, acronym_list, subject_num)
train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)

torch.save(subject_od_ind, save_dir + f'/Model/Allen/subject_od_ind_Allen_{key}{0}.pt')
torch.save(subject_od_list, save_dir + f'/Model/Allen/subject_od_list_Allen_{key}{0}.pt')

data_train = TensorDataset(brain_signal_lfp[train_ind,:], brain_region_index[train_ind], coordinate_list[train_ind])
data_valid = TensorDataset(brain_signal_lfp[valid_ind,:], brain_region_index[valid_ind], coordinate_list[valid_ind])
data_test = TensorDataset(brain_signal_lfp[test_ind,:], brain_region_index[test_ind], coordinate_list[test_ind])

train_iter = DataLoader(data_train, batch_size=128, shuffle=True)
valid_iter = DataLoader(data_valid, batch_size=128, shuffle=True)
test_iter = DataLoader(data_test, batch_size=128, shuffle=True)

FRP1
FRP2/3
FRP5
FRP6a
MOp1
MOp2/3
MOp5
MOp6a
MOp6b
MOs1
MOs2/3
MOs5
MOs6a
MOs6b
SSp-n1
SSp-n2/3
SSp-n4
SSp-n5
SSp-n6a
SSp-n6b
SSp-bfd1
SSp-bfd2/3
SSp-bfd4
SSp-bfd5
SSp-bfd6a
SSp-bfd6b
SSp-ll1
SSp-ll2/3
SSp-ll4
SSp-ll5
SSp-ll6a
SSp-ll6b
SSp-m1
SSp-m2/3
SSp-m4
SSp-m5
SSp-m6a
SSp-m6b
SSp-ul1
SSp-ul2/3
SSp-ul4
SSp-ul5
SSp-ul6a
SSp-ul6b
SSp-tr1
SSp-tr2/3
SSp-tr4
SSp-tr5
SSp-tr6a
SSp-tr6b
SSp-un1
SSp-un2/3
SSp-un4
SSp-un5
SSp-un6a
SSp-un6b
SSs2/3
SSs4
SSs5
SSs6a
SSs6b
GU5
GU6a
VISC5
VISC6a
VISC6b
AUDd2/3
AUDd4
AUDd5
AUDd6a
AUDd6b
AUDp4
AUDp5
AUDp6a
AUDpo2/3
AUDpo4
AUDpo5
AUDpo6a
AUDpo6b
AUDv5
AUDv6a
AUDv6b
VISal2/3
VISal4
VISal5
VISal6a
VISal6b
VISam1
VISam2/3
VISam4
VISam5
VISam6a
VISam6b
VISl1
VISl2/3
VISl4
VISl5
VISl6a
VISl6b
VISp1
VISp2/3
VISp4
VISp5
VISp6a
VISp6b
VISpl1
VISpl2/3
VISpl4
VISpl5
VISpl6a
VISpm1
VISpm2/3
VISpm4
VISpm5
VISpm6a
VISpm6b
VISli2/3
VISli4
VISli5
VISli6a
VISli6b
VISpor1
VISpor2/3
VISpor5
VISpor6a
VISpor6b
ACAd2/3
ACAd5
ACAd6a
ACAd6b
ACAv1
ACAv2/3
ACAv5
ACAv6a
ACAv

In [ ]:
c0 = 64 * 4
k0 = 1.0

model_args = {
    'arch':((2,c0 * 2,1,k0), (2,c0 * 3,1,k0), (2,c0 * 4,1,k0), (2,c0 * 5,1,k0)),
    'stem_channels':c0,
}
train_args = {
    'overfitting_thres':0.60,
    'lr':5e-4,
    'norm':True,
    'temp':[True, True],
    'epochs':50,
    'save_dir':save_dir,
}

for ind in range(0, 1):
    Classifier = AnyNet_L(model_args['arch'], model_args['stem_channels'], frequency_bin=spectro_args['spectro_img'][0], num_classes=len(acronym_list)).to(device)
    Classifier.apply(init_cnn)
    net_train_AnyNet_L(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


accu_fun is correct>>>>>>>>>>>>>>>>>>>>>>>>>>>>
tensor(0.0078)


/usr/local/lib/python3.10/dist-packages/torch/autograd/graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


tensor(0.0547)
tensor(0.1094)
tensor(0.1406)
tensor(0.2266)
tensor(0.3516)
tensor(0.3125)
tensor(0.3125)
tensor(0.3125)
tensor(0.3281)
tensor(0.3516)
tensor(0.3516)
tensor(0.3750)
Acu Train: 0.26135149598121643
Acu valid: 0.31784185767173767
tensor(0.4141)
tensor(0.3516)
tensor(0.4531)
tensor(0.3672)
tensor(0.4922)
tensor(0.4531)
tensor(0.4453)
tensor(0.5391)
tensor(0.4375)
tensor(0.4297)
tensor(0.4219)
tensor(0.4766)
tensor(0.4688)
Acu Train: 0.4505269229412079
Acu valid: 0.3626876771450043
tensor(0.4453)
tensor(0.4922)
tensor(0.5000)
tensor(0.5547)
tensor(0.4609)
tensor(0.5547)
tensor(0.5234)
tensor(0.5625)
tensor(0.5625)
tensor(0.5156)
tensor(0.6250)
tensor(0.5625)
tensor(0.6094)
Acu Train: 0.5565057396888733
Acu valid: 0.3406549096107483
tensor(0.6797)
tensor(0.6250)
tensor(0.6797)
tensor(0.6562)
tensor(0.7109)
tensor(0.6250)
tensor(0.6484)
tensor(0.5469)
tensor(0.6797)
tensor(0.6250)
tensor(0.6016)
tensor(0.7031)
tensor(0.6328)
Acu Train: 0.6398121118545532
Acu valid: 0.3476413786

In [ ]:
for ind in range(0, 1):
    img_size, patch_size = (224, 28), (28, 28)
    num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 4
    emb_dropout, blk_dropout = 0.1, 0.1
    Classifier = ViT_L(spectro_args['spectro_img'][0], img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout, num_classes=len(acronym_list)).to(device)
    net_train_ViT_L(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,


tensor(0.)
tensor(0.0312)
tensor(0.0859)
tensor(0.0859)
tensor(0.0625)
tensor(0.1016)
tensor(0.1328)
tensor(0.1797)
tensor(0.1484)
tensor(0.1094)
tensor(0.1250)
tensor(0.1797)
tensor(0.1172)
Acu Train: 0.11258242279291153
Acu valid: 0.16715669631958008
tensor(0.1484)
tensor(0.1016)
tensor(0.1641)
tensor(0.1016)
tensor(0.1562)
tensor(0.1953)
tensor(0.2969)
tensor(0.2578)
tensor(0.2031)
tensor(0.2188)
tensor(0.1719)
tensor(0.2031)
tensor(0.3047)
Acu Train: 0.20952244102954865
Acu valid: 0.24191346764564514
tensor(0.2188)
tensor(0.2422)
tensor(0.2734)
tensor(0.2969)
tensor(0.3125)
tensor(0.3203)
tensor(0.2734)
tensor(0.2266)
tensor(0.2344)
tensor(0.2969)
tensor(0.3125)
tensor(0.2891)
tensor(0.2734)
Acu Train: 0.25939539074897766
Acu valid: 0.2774638533592224
tensor(0.3594)
tensor(0.3438)
tensor(0.2656)
tensor(0.3047)
tensor(0.2734)
tensor(0.3359)
tensor(0.2578)
tensor(0.3594)
tensor(0.2891)
tensor(0.3438)
tensor(0.3359)
tensor(0.3516)
tensor(0.3828)
Acu Train: 0.30225881934165955
Acu vali

In [ ]:
RNN_args = {
    'input_size':224,
    'hidden_size':512 * 2,
    'num_layers':2,
    'category_num':len(acronym_list),
    'data_len':28,
}
for ind in range(0, 1):
    Classifier = RNN_L(spectro_args['spectro_img'][0], RNN_args).to(device)
    net_train_RNN_L(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

tensor(0.)
tensor(0.0703)
tensor(0.1250)
tensor(0.1562)
tensor(0.1328)
tensor(0.1172)
tensor(0.1641)
tensor(0.2734)
tensor(0.2188)
tensor(0.2500)
tensor(0.2266)
tensor(0.3672)
tensor(0.2891)
Acu Train: 0.21656644344329834
Acu valid: 0.24850605428218842
tensor(0.3203)
tensor(0.3516)
tensor(0.3984)
tensor(0.4141)
tensor(0.3984)
tensor(0.3359)
tensor(0.4219)
tensor(0.3828)
tensor(0.4297)
tensor(0.4375)
tensor(0.4922)
tensor(0.4609)
tensor(0.4609)
Acu Train: 0.41797396540641785
Acu valid: 0.2805498540401459
tensor(0.4922)
tensor(0.4766)
tensor(0.4297)
tensor(0.5156)
tensor(0.5469)
tensor(0.5859)
tensor(0.4219)
tensor(0.5469)
tensor(0.4453)
tensor(0.5547)
tensor(0.5234)
tensor(0.4922)
tensor(0.5781)
Acu Train: 0.5284436345100403
Acu valid: 0.29372256994247437
tensor(0.5938)
tensor(0.5469)
tensor(0.6250)
tensor(0.6094)
tensor(0.5469)
tensor(0.5938)
tensor(0.6641)
tensor(0.6406)
tensor(0.6562)
tensor(0.6406)
tensor(0.6328)
tensor(0.6562)
tensor(0.5859)
Acu Train: 0.6043452024459839
Acu valid:

In [ ]:
LC_args = {
    'category_num':len(acronym_list),
}
Classifier = LinearClassifier(spectro_args['spectro_img'][0], LC_args).to(device)

torch.save(Classifier, train_args['save_dir'] + f'/Model/Allen/LC_L_Allen_chance_{key}_0.pth')

In [ ]:
for ind in range(0, 1):
    Classifier = LinearClassifier(spectro_args['spectro_img'][0], LC_args).to(device)
    net_train_LC_L(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

tensor(0.)
tensor(0.0469)
tensor(0.0391)
tensor(0.0391)
tensor(0.0234)
tensor(0.0234)
tensor(0.0938)
tensor(0.0469)
tensor(0.0469)
tensor(0.0625)
tensor(0.0625)
tensor(0.0625)
tensor(0.0625)
Acu Train: 0.051687851548194885
Acu valid: 0.059773679822683334
tensor(0.0547)
tensor(0.0469)
tensor(0.0703)
tensor(0.0547)
tensor(0.0625)
tensor(0.0625)
tensor(0.0781)
tensor(0.0625)
tensor(0.0469)
tensor(0.0625)
tensor(0.0547)
tensor(0.0703)
tensor(0.0781)
Acu Train: 0.06903310120105743
Acu valid: 0.07118892669677734
tensor(0.0703)
tensor(0.0938)
tensor(0.0781)
tensor(0.0938)
tensor(0.1172)
tensor(0.0859)
tensor(0.0469)
tensor(0.0703)
tensor(0.0703)
tensor(0.0859)
tensor(0.0859)
tensor(0.0859)
tensor(0.1094)
Acu Train: 0.07895062863826752
Acu valid: 0.07856659591197968
tensor(0.0625)
tensor(0.0859)
tensor(0.0469)
tensor(0.1484)
tensor(0.0625)
tensor(0.0781)
tensor(0.0391)
tensor(0.0391)
tensor(0.0547)
tensor(0.1172)
tensor(0.1094)
tensor(0.0859)
tensor(0.0781)
Acu Train: 0.08600782603025436
Acu v

In [ ]:
from google.colab import runtime
runtime.unassign()